In [ ]:
!pip install -q -U langchain langchain-core langchain-groq langchain-community langchain-chroma langchain-huggingface langchain-text-splitters
!pip install -q chromadb sentence-transformers gradio

In [ ]:
from google.colab import userdata, files
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Get your Groq API key from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Add GROQ_API_KEY in Google Colab Secrets first."
    )

print("Libraries imported and Groq API key loaded!")

In [ ]:
uploaded = files.upload()

file_path = next(iter(uploaded.keys()))

loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print("Document loaded successfully!")
print(documents[0].page_content[:1000])

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0].page_content)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_documents"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Vector database created successfully!")

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0
)

print("LLM configured successfully!")

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful document question-answering assistant.

Answer the question using only the context below.
If the answer is not available in the context, say:
"I couldn't find the answer in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully!")

In [ ]:
def ask_rag(question):
    if not question.strip():
        return "Please enter a question."

    answer = rag_chain.invoke(question)
    return answer

question = "What is the main topic of this document?"
print(ask_rag(question))

In [ ]:

import gradio as gr
import uuid
import html
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# RAG DOCUMENT CHATBOT — GRADIO 6 COMPATIBLE
# Requires existing variables: llm, embeddings
# ============================================================

vectorstore = None
retriever = None
current_collection = None

DEFAULT_INSTRUCTION = (
    "Answer using only the supplied document context. "
    "If the answer is missing, say you could not find it "
    "in the uploaded document. Keep the answer clear and concise."
)

FIVE_QUESTIONS = [
    "What is the main topic of this document?",
    "What are the key points discussed?",
    "Explain the most important concept.",
    "What examples or facts are mentioned?",
    "Summarize the document in simple words."
]


# ============================================================
# 1. UPLOAD A TXT DOCUMENT AND CREATE VECTOR STORE
# ============================================================

def upload_document(file):
    global vectorstore, retriever, current_collection

    if not file:
        return "Please upload a .txt file.", "No document loaded."

    try:
        # Gradio may provide a filepath or a file object.
        file_path = file if isinstance(file, str) else file.name

        if not file_path.lower().endswith(".txt"):
            return "Please upload a .txt file.", "Invalid file type."

        documents = TextLoader(
            file_path,
            encoding="utf-8"
        ).load()

        if not documents or not documents[0].page_content.strip():
            return "The uploaded file is empty.", "No document loaded."

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=300,
            chunk_overlap=50
        )

        chunks = splitter.split_documents(documents)

        if not chunks:
            return "No text chunks could be created.", "Upload failed."

        collection_name = "rag_" + uuid.uuid4().hex[:12]

        new_vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=collection_name
        )

        # Update the active store only after successful creation.
        vectorstore = new_vectorstore
        current_collection = collection_name

        retriever = vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        filename = os.path.basename(file_path)

        return (
            f"Document loaded: {filename}\n"
            f"Chunks created: {len(chunks)}\n"
            f"Collection: {collection_name}",
            f"Ready | Retriever k = 3 | {filename}"
        )

    except Exception as error:
        return (
            f"Upload failed: {error}",
            "Upload failed. Check the file and embedding setup."
        )


# ============================================================
# 2. RETRIEVER SETTINGS
# ============================================================

def update_retriever_k(k):
    global retriever

    if vectorstore is None:
        return "Upload a document first."

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": int(k)}
    )

    return f"Retriever updated: k = {int(k)}"


# ============================================================
# SHARED RAG QUESTION-ANSWERING FUNCTION
# ============================================================

def answer_question(question, k, instruction):
    if vectorstore is None:
        return "Please upload a document first."

    if not question or not question.strip():
        return "Please enter a question."

    docs = vectorstore.as_retriever(
        search_kwargs={"k": int(k)}
    ).invoke(question.strip())

    if not docs:
        return "No relevant information was found in the document."

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    chosen_instruction = (
        instruction.strip()
        if instruction and instruction.strip()
        else DEFAULT_INSTRUCTION
    )

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "{instruction}\n\n"
            "Document context:\n{context}"
        ),
        ("human", "{question}")
    ])

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({
        "instruction": chosen_instruction,
        "context": context,
        "question": question.strip()
    })


# ============================================================
# 3. COMPARE RETRIEVER k=1 VS k=3
# ============================================================

def compare_k_values(question, instruction):
    if not question or not question.strip():
        return "Enter a question first.", "Enter a question first."

    if vectorstore is None:
        return "Upload a document first.", "Upload a document first."

    try:
        answer_k1 = answer_question(question, 1, instruction)
        answer_k3 = answer_question(question, 3, instruction)

        return answer_k1, answer_k3

    except Exception as error:
        message = f"Comparison failed: {error}"
        return message, message


# ============================================================
# 4. DOCUMENT CHAT — MESSAGE HISTORY FOR GRADIO 6
# Do not pass type="messages" to gr.Chatbot().
# ============================================================

def chat_with_document(question, history, k, instruction):
    history = history or []

    if not question or not question.strip():
        return history, ""

    try:
        answer = answer_question(question, k, instruction)
    except Exception as error:
        answer = f"Unable to answer: {error}"

    # Gradio 6 message format
    history = history + [
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": answer}
    ]

    return history, ""


# ============================================================
# 5. CUSTOMIZE WEBPAGE TITLE
# ============================================================

def update_page_title(title):
    safe_title = html.escape(
        (title or "").strip() or "RAG Document Chatbot"
    )

    return f"""
    <div class="hero">
        <div class="hero-icon">📄</div>
        <h1>{safe_title}</h1>
        <p>Upload a text document and ask questions based on its content.</p>
        <div class="badges">
            <span>📘 RAG</span>
            <span>🔗 LangChain + Chroma</span>
            <span>⚡ Groq LLM</span>
            <span>🤖 Powered by AI</span>
        </div>
    </div>
    """


# ============================================================
# 6. CUSTOM EXAMPLE QUESTIONS
# ============================================================

def update_examples(text):
    questions = [
        line.strip()
        for line in (text or "").splitlines()
        if line.strip()
    ]

    return questions[:10] if questions else FIVE_QUESTIONS


def examples_count(text):
    questions = update_examples(text)
    return f"{len(questions)} example questions configured."


# ============================================================
# 7. SHOW RETRIEVED DOCUMENT CONTEXT
# ============================================================

def retrieve_context(question, k):
    if vectorstore is None:
        return "Upload a document first."

    if not question or not question.strip():
        return "Enter a question first."

    try:
        docs = vectorstore.as_retriever(
            search_kwargs={"k": int(k)}
        ).invoke(question.strip())

        if not docs:
            return "No relevant chunks found."

        return "\n\n".join(
            f"Chunk {i + 1}:\n{doc.page_content}"
            for i, doc in enumerate(docs)
        )

    except Exception as error:
        return f"Context retrieval failed: {error}"


# ============================================================
# CSS DESIGN
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
    background: #f5f8ff !important;
    font-family: Inter, Arial, sans-serif !important;
}

body {
    background: #f5f8ff !important;
}

footer {
    display: none !important;
}

.hero {
    text-align: center;
    padding: 28px 12px;
    margin-bottom: 20px;
    background: linear-gradient(135deg, #eef5ff, #f9faff);
    border: 1px solid #dce6ff;
    border-radius: 18px;
}

.hero-icon {
    font-size: 38px;
}

.hero h1 {
    color: #14265d;
    font-size: clamp(28px, 3vw, 42px);
    margin: 8px 0;
}

.hero p {
    color: #50658c;
    font-size: 16px;
}

.badges {
    display: flex;
    flex-wrap: wrap;
    justify-content: center;
    gap: 10px;
    margin-top: 20px;
}

.badges span {
    padding: 9px 14px;
    border-radius: 30px;
    background: #edf0ff;
    color: #4037d8;
    border: 1px solid #d8dcff;
    font-size: 13px;
}

.panel {
    min-width: 0 !important;
    background: white !important;
    border: 1px solid #dce5ff !important;
    border-radius: 15px !important;
    padding: 20px !important;
    box-shadow: 0 5px 18px rgba(46,73,150,.05);
}

.section-title {
    color: #14265d;
    font-size: 20px;
    font-weight: 700;
}

.helper {
    color: #50658c;
    line-height: 1.6;
    font-size: 14px;
}

#chatbot {
    border: 1px solid #e0e8ff !important;
    border-radius: 12px !important;
    background: white !important;
}

#upload-btn button,
#send-btn button,
#compare-btn button,
#update-title-btn button,
#context-btn button {
    background: linear-gradient(110deg, #5c5af7, #5149e8) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}

#footer-note {
    text-align: center;
    color: #6b7d9d;
    padding: 24px 10px;
    font-size: 13px;
}

@media (max-width: 768px) {
    .panel {
        padding: 14px !important;
    }
}
"""


# ============================================================
# BUILD THE GRADIO WEBPAGE
# ============================================================

with gr.Blocks(title="RAG Document Chatbot") as demo:

    page_header = gr.HTML(
        value=update_page_title("RAG Document Chatbot")
    )

    with gr.Row(equal_height=True):

        # ----------------------------------------------------
        # LEFT PANEL
        # ----------------------------------------------------

        with gr.Column(scale=4, elem_classes=["panel"]):

            gr.HTML("""
                <div class="section-title">
                    📄 1. Upload Your Document
                </div>
                <p class="helper">
                    Upload a different .txt file to test the RAG pipeline.
                </p>
            """)

            file_input = gr.File(
                label="Choose a .txt file",
                file_types=[".txt"],
                type="filepath"
            )

            upload_button = gr.Button(
                "📂 Upload Document",
                elem_id="upload-btn"
            )

            upload_status = gr.Textbox(
                label="Document Status",
                value="No document uploaded yet.",
                interactive=False,
                lines=3
            )

            gr.HTML("<hr>")

            gr.HTML("""
                <div class="section-title">
                    🔍 2. Retriever Comparison
                </div>
                <p class="helper">
                    Compare answers using k=1 and k=3.
                </p>
            """)

            k_selector = gr.Radio(
                choices=[1, 3],
                value=3,
                label="Retrieved chunks (k)"
            )

            k_status = gr.Textbox(
                label="Retriever Status",
                value="Default: k = 3",
                interactive=False
            )

            compare_question = gr.Textbox(
                label="Comparison question",
                placeholder="Enter the same question for both settings..."
            )

            compare_button = gr.Button(
                "⚖️ Compare k=1 vs k=3",
                elem_id="compare-btn"
            )

            answer_k1 = gr.Textbox(
                label="Answer with k=1",
                lines=4,
                interactive=False
            )

            answer_k3 = gr.Textbox(
                label="Answer with k=3",
                lines=4,
                interactive=False
            )

            gr.HTML("<hr>")

            gr.HTML("""
                <div class="section-title">
                    🛠️ 3. Change Prompt Instruction
                </div>
                <p class="helper">
                    Edit the instruction and ask the same question again.
                </p>
            """)

            instruction_input = gr.Textbox(
                label="System prompt",
                value=DEFAULT_INSTRUCTION,
                lines=4
            )

            gr.HTML("<hr>")

            gr.HTML("""
                <div class="section-title">
                    🎨 4. Customize Title and Examples
                </div>
            """)

            title_input = gr.Textbox(
                label="Webpage title",
                value="RAG Document Chatbot"
            )

            examples_input = gr.Textbox(
                label="Example questions (one per line)",
                value="\n".join(FIVE_QUESTIONS),
                lines=5
            )

            update_title_button = gr.Button(
                "✨ Update Title",
                elem_id="update-title-btn"
            )

            examples_status = gr.Textbox(
                label="Example question status",
                value="5 example questions configured.",
                interactive=False
            )

        # ----------------------------------------------------
        # RIGHT PANEL
        # ----------------------------------------------------

        with gr.Column(scale=6, elem_classes=["panel"]):

            gr.HTML("""
                <div class="section-title">💬 Document Chat</div>
                <p class="helper">
                    Upload a document first, then ask questions.
                </p>
            """)

            # Gradio 6: omit the unsupported type="messages" argument.
            chatbot = gr.Chatbot(
                label="Conversation",
                height=430,
                elem_id="chatbot"
            )

            question_input = gr.Textbox(
                label="Ask your document",
                placeholder="Type your question here...",
                lines=2
            )

            with gr.Row():
                send_button = gr.Button(
                    "➤ Send",
                    elem_id="send-btn"
                )

                clear_button = gr.Button("🗑️ Clear Chat")

            gr.Markdown("### 💡 Five Example Questions")

            gr.Examples(
                examples=[[q] for q in FIVE_QUESTIONS],
                inputs=question_input,
                label="Click a question to try it"
            )

            gr.Markdown("### 📚 Retrieved Context Preview")

            context_question = gr.Textbox(
                label="Question for context preview",
                placeholder="Enter a question to inspect retrieved chunks"
            )

            context_button = gr.Button(
                "Show Retrieved Context",
                elem_id="context-btn"
            )

            context_output = gr.Textbox(
                label="Retrieved document chunks",
                lines=10,
                interactive=False
            )

    gr.HTML("""
        <div id="footer-note">
            RAG Document Chatbot · LangChain · Chroma · Groq · Gradio
            <br>Day 6 RAG Exercises
        </div>
    """)

    # ========================================================
    # EVENT HANDLERS
    # ========================================================

    upload_button.click(
        fn=upload_document,
        inputs=file_input,
        outputs=[upload_status, k_status]
    )

    k_selector.change(
        fn=update_retriever_k,
        inputs=k_selector,
        outputs=k_status
    )

    compare_button.click(
        fn=compare_k_values,
        inputs=[compare_question, instruction_input],
        outputs=[answer_k1, answer_k3]
    )

    send_button.click(
        fn=chat_with_document,
        inputs=[
            question_input,
            chatbot,
            k_selector,
            instruction_input
        ],
        outputs=[chatbot, question_input]
    )

    question_input.submit(
        fn=chat_with_document,
        inputs=[
            question_input,
            chatbot,
            k_selector,
            instruction_input
        ],
        outputs=[chatbot, question_input]
    )

    clear_button.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot
    )

    context_button.click(
        fn=retrieve_context,
        inputs=[context_question, k_selector],
        outputs=context_output
    )

    update_title_button.click(
        fn=update_page_title,
        inputs=title_input,
        outputs=page_header
    )

    examples_input.change(
        fn=examples_count,
        inputs=examples_input,
        outputs=examples_status
    )


# ============================================================
# LAUNCH — GRADIO 6
# ============================================================

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
)


In [ ]:
import gradio as gr
print(gr.__version__)